## 1. 导入 API

In [1]:
from pathlib import Path
import json

from InterOptimus.agents.simple_iomaker import run_simple_iomaker
from InterOptimus.agents.remote_submit import iomaker_status, iomaker_fetch_results

## 2. 准备一个本地配置

新格式由四个顶层部分组成：

- `workflow_name`: 本次任务名称。
- `IO_workflow_config`: 物理与计算参数。
- `execution`: 本地运行还是 server 提交。
- `cluster`: server / Slurm 相关设置，本地运行可以留空。

下面的例子使用 `execution = "server"`，在"interactive" cpu节点上运行mlip，也在同节点运行VASP

In [3]:
film_cif = Path("film.cif")
substrate_cif = Path("substrate.cif")

config = {
    "workflow_name": "IO_test",
    "execution": "server",  
    "IO_workflow_config": {
                            "bulk_cifs": {"film_cif": film_cif,
                                          "substrate_cif": substrate_cif},
                            "lattice_matching_settings": {"max_area": 20,
                                                            "max_length_tol": 0.1,
                                                            "max_angle_tol": 0.1,
                                                            "film_max_miller": 3,
                                                            "substrate_max_miller": 3},
                            "structure_settings": {"termination_ftol": 0.15,
                                                        "film_thickness": 6,
                                                        "substrate_thickness": 6,
                                                        "double_interface": True},
                            "optimization_settings": {"fmax": 0.05,
                                                    "steps": 500,
                                                    "device": "cpu",
                                                    "discut": 0.8,
                                                    "n_calls_density": 1,
                                                    "z_range": [0.5, 3.0],
                                                    "calc": "orb-models",},
                            "vasp_settings": {"do_vasp": True,
                                                "static_user_incar_settings": {"ALGO":"Fast"},
                                                "relax_user_incar_settings": {"ALGO":"Fast"},
                                                "relax_user_potcar_functional": "PBE_54",
                                                "static_user_potcar_functional": "PBE_54"}
                            },
    "cluster": {
        "mlip":{"mlip_worker":"std_worker",
                "slurm_partition":"interactive",
                "mlip_project":"std",
               "mlip_cpus_per_task":16},
        "vasp": {"vasp_worker":"std_worker",
                "vasp_slurm_partition": "interactive",
                "vasp_nodes": 1,
                "vasp_processes_per_node": 48,
                "vasp_pre_run": "module load VASP/6.5.0",
                }}
}

# 若采用gpu,"optimization_settings"中"device"换成cuda，"cluster"中"mlip"下"mlip_cpus_per_task"换成"cpus_per_gpu"

## 3. 执行

In [22]:
result = run_simple_iomaker(config)


✅ InterOptimus IOMaker job generated
📄 Flow JSON: /home/xys/software/InterOptimus/examples/IO_test/io_flow.json
📦 Structures source: local_cif
🧱 film.cif: /home/xys/software/InterOptimus/examples/film.cif
🧱 substrate.cif: /home/xys/software/InterOptimus/examples/substrate.cif

⚙️  Jobflow configuration:
   MLIP worker: std_worker
   MLIP project: std
   MLIP resources: ✅ Set
   VASP worker: std_worker
   VASP resources: ✅ Set
   VASP GD: ❌ Off
   VASP exec_config (pre_run, …): {'pre_run': 'module load VASP/6.5.0'}

🧪 VASP input (InterfaceWorker.patch_jobflow_jobs / itworker-style):
{
  "relax_user_incar_settings": {
    "ALGO": "Fast"
  },
  "relax_user_potcar_settings": null,
  "relax_user_kpoints_settings": null,
  "relax_user_potcar_functional": "PBE_54",
  "static_user_incar_settings": {
    "ALGO": "Fast"
  },
  "static_user_potcar_settings": null,
  "static_user_kpoints_settings": null,
  "static_user_potcar_functional": "PBE_54",
  "gd_kwargs": {
    "tol": 0.0005
  },
  "dipol

记住下面UUID，后续如果关闭这个进程后需要通过这个UUID访问结果

In [34]:
result['mlip_job_uuid']

'0258c415-664d-47f3-b235-8c1190a337ee'

## 4. 查看状态

用上一步UUID查询

In [38]:
status = iomaker_status('0258c415-664d-47f3-b235-8c1190a337ee')

job_id: 0258c415-664d-47f3-b235-8c1190a337ee
 phase: VASP 阶段 | mode: mlip_and_vasp
 mlip_state: COMPLETED
 summary: 当前为 VASP 阶段。 真实 VASP 子任务 2/4 已结束； completed=2, running=2, pending=0, failed=0.
 vasp_jobs: 2/4 finished (completed=2, running=2, pending=0, failed=0)
 vasp_sub_jobs:
  - fa655774-5d03-442d-a8ec-d72fb37e8b14 state= COMPLETED 
  - 62fc8154-4abc-44d8-81d7-8c76c90f33ae state= COMPLETED 
  - 5d762cec-8137-4990-adb1-2c3858836a34 state= RUNNING 
  - f29c21be-a1db-434e-b8b5-cbfde5e87ebb state= WAITING 


## 5. 拉取结果

用UUID获取

In [3]:
fetch_results = iomaker_fetch_results('0258c415-664d-47f3-b235-8c1190a337ee')

/home/xys/software/InterOptimus/InterOptimus/matching.py:2061: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.85, 1], pad=0.1)


job_id: 0258c415-664d-47f3-b235-8c1190a337ee
 export_dir: /home/xys/software/InterOptimus/examples/0258c415_results
 success: True  run_dir: /home/xys/std_jobs/02/58/c4/0258c415-664d-47f3-b235-8c1190a337ee_1
 summary: 当前为 全部结束。 真实 VASP 子任务 4/4 已结束； completed=4, running=0, pending=0, failed=0.
 vasp_results: /home/xys/software/InterOptimus/examples/0258c415_results/vasp_results
 mlip_results: /home/xys/software/InterOptimus/examples/0258c415_results/mlip_results
 vasp stereographic: /home/xys/software/InterOptimus/examples/0258c415_results/vasp_results/stereographic.jpg
